# LCM-UNet — Colab Runner (the only notebook you open in Colab)

This notebook is the **bridge**. It never contains hand-copied model/training
code — every run it pulls the latest code fresh from GitHub, so there is
never a divergence between what's on GitHub and what runs here.

Run All top to bottom each session. If the session dies mid-run, just
reopen this notebook and Run All again — `run_all_pending()` resumes the job
queue from Google Drive with zero manual state (see `lcmunet/run_manifest.py`).


In [ ]:
# (a) Mount Google Drive — this is the persistence layer for everything
# (checkpoints/splits/configs/logs/results/figures).
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# (b) git clone/pull THIS repo into /content/ — no code is ever hand-copied.
import os
import subprocess

REPO_URL = "https://github.com/Shihabul-Shuvo/LCM-UNet.git"
REPO_DIR = "/content/LCM-UNet"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# (c) copy secrets/kaggle.json -> ~/.kaggle/ if present on Drive.
import shutil
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/LCM-UNet/"
kaggle_src = Path(DRIVE_ROOT) / "secrets" / "kaggle.json"
kaggle_dst_dir = Path.home() / ".kaggle"

if kaggle_src.exists():
    kaggle_dst_dir.mkdir(parents=True, exist_ok=True)
    dst = kaggle_dst_dir / "kaggle.json"
    shutil.copy(kaggle_src, dst)
    dst.chmod(0o600)
    print("kaggle.json copied to", dst)
else:
    print("No secrets/kaggle.json found on Drive yet; skipping Kaggle auth setup.")


In [ ]:
# (d) pip install requirements (infra-layer deps only; torch is Colab's own).
%pip install -q -r requirements.txt


In [ ]:
# (e) set DRIVE_ROOT so lcmunet.paths.resolve_drive_root() picks it up.
import os
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
print("DRIVE_ROOT =", os.environ["DRIVE_ROOT"])


In [ ]:
# (f) print GPU name and free VRAM — GPU gates are confirmed by YOU pasting
# this output back, never claimed by the agent.
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info(0)
    print(f"GPU: {name}")
    print(f"Free VRAM:  {free_b / 1024**3:.2f} GB")
    print(f"Total VRAM: {total_b / 1024**3:.2f} GB")
else:
    print("WARNING: no CUDA GPU detected. Runtime > Change runtime type > GPU, then Run All again.")


In [ ]:
# (g) single call that drives the job queue (results/manifest.json on Drive).
# Training/runner logic does not exist yet (infra-layer only, per methodology
# Week 1); run_all_pending() will raise NotImplementedError until it does.
from lcmunet.paths import get_paths
from lcmunet.run_manifest import run_queue


def run_all_pending(max_minutes: float = 300.0) -> None:
    """Process PENDING jobs in results/manifest.json until empty or time runs out.

    Safe to call from a brand-new Colab session after a disconnect: stale
    RUNNING jobs (killed by the previous session) are reclaimed automatically.
    """
    paths = get_paths()

    def runner_fn(job):
        raise NotImplementedError(
            "No training entrypoint yet — infra layer only (Week 1). "
            f"Job requested: {job['config_id']} -> {job['config_yaml_path']}"
        )

    run_queue(paths.results, runner_fn, max_minutes=max_minutes)


print("Bridge ready. Call run_all_pending() once the training entrypoint exists.")
